# DWM Lab 9 — Apriori Association Rule Mining

### Dataset: Online Retail

**Objective:**  
To implement the **Apriori algorithm** for frequent itemset mining and generate association rules from the Online Retail transaction dataset.

The experiment includes:

- Transaction data preprocessing
- Basket/transaction creation
- One-hot encoding
- Frequent itemset generation
- Support calculation
- Association rule generation
- Confidence calculation
- Lift calculation
- Filtering and interpretation of rules

### Dataset adaptation

The same **Online Retail.xlsx** dataset is used for this experiment.

Each `InvoiceNo` represents a transaction and `Description` represents a purchased product. Therefore, the data can naturally be converted into a transaction-item format required for Apriori.

## 1. Import Required Libraries

Pandas is used for data manipulation, while `mlxtend` provides implementations of Apriori and association-rule mining.

If `mlxtend` is not installed, uncomment and run the installation cell below.

In [ ]:
# Run this cell only if mlxtend is not installed
# %pip install mlxtend openpyxl

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from mlxtend.frequent_patterns import apriori, association_rules

## 2. Load the Dataset

Load the same `Online Retail.xlsx` dataset used throughout the DWM experiments.

In [ ]:
file_path = "Online Retail.xlsx"
df = pd.read_excel(file_path)

print("Dataset shape:", df.shape)
display(df.head())

## 3. Data Preprocessing

For association rule mining:

1. Remove cancelled invoices.
2. Remove transactions with non-positive quantities.
3. Remove rows with missing product descriptions.
4. Keep valid invoice and product information.

Cancelled invoices are excluded because they do not represent completed purchases.

In [ ]:
# Remove cancelled invoices
df = df[~df['InvoiceNo'].astype(str).str.startswith('C')].copy()

# Keep valid quantities
df = df[df['Quantity'] > 0].copy()

# Remove missing product descriptions
df = df.dropna(subset=['Description'])

# Clean product descriptions
df['Description'] = df['Description'].astype(str).str.strip()

# Remove empty descriptions
df = df[df['Description'] != ''].copy()

print("Cleaned dataset shape:", df.shape)
print("Unique invoices:", df['InvoiceNo'].nunique())
print("Unique products:", df['Description'].nunique())

display(df[['InvoiceNo', 'Description', 'Quantity']].head())

## 4. Understand the Transaction Structure

Apriori works on a set of transactions.

Example:

| Invoice | Items |
|---|---|
| 10001 | Bread, Milk, Butter |
| 10002 | Milk, Eggs |
| 10003 | Bread, Milk |

From this structure, Apriori finds products that frequently occur together.

In this dataset:
- **Transaction ID = InvoiceNo**
- **Item = Description**

In [ ]:
# Number of items per invoice
items_per_invoice = df.groupby('InvoiceNo')['Description'].nunique()

print("Average unique products per invoice:",
      round(items_per_invoice.mean(), 2))

print("\nInvoices with at least 2 different products:",
      (items_per_invoice >= 2).sum())

print("\nTop 10 products by number of invoices:")
top_products = (
    df.groupby('Description')['InvoiceNo']
    .nunique()
    .sort_values(ascending=False)
    .head(10)
)

display(top_products)

## 5. Reduce the Item Space for Apriori

The Online Retail dataset contains many different products. Running Apriori on every product can create a very large number of candidate itemsets.

For a practical lab execution, the **50 most frequently purchased products** are selected.

This does not change the Apriori algorithm; it makes the experiment computationally manageable.

In [ ]:
TOP_N_ITEMS = 50

top_item_names = (
    df.groupby('Description')['InvoiceNo']
    .nunique()
    .sort_values(ascending=False)
    .head(TOP_N_ITEMS)
    .index
)

apriori_df = df[df['Description'].isin(top_item_names)].copy()

print("Products retained:", apriori_df['Description'].nunique())
print("Invoices represented:", apriori_df['InvoiceNo'].nunique())

display(
    apriori_df[['InvoiceNo', 'Description', 'Quantity']].head()
)

## 6. Create the Transaction Basket

Create a basket matrix where:

- Rows = invoices
- Columns = products
- Value `1` = product was purchased
- Value `0` = product was not purchased

This is the one-hot encoded format required by Apriori.

In [ ]:
basket = (
    apriori_df
    .groupby(['InvoiceNo', 'Description'])['Quantity']
    .sum()
    .unstack(fill_value=0)
)

# Convert quantities to 0/1 presence values
basket = (basket > 0).astype(int)

print("Basket shape:", basket.shape)
display(basket.head())

## 7. Transaction Example

Display the products purchased in the first few transactions.

In [ ]:
for invoice in basket.index[:5]:
    purchased_items = basket.columns[basket.loc[invoice] == 1].tolist()
    print(f"Invoice {invoice}:")
    print(purchased_items)
    print()

## 8. Apriori Algorithm

### Apriori principle

The Apriori algorithm identifies frequent itemsets using a minimum support threshold.

**Support:**

Support(X) = Number of transactions containing X / Total number of transactions

If an itemset is infrequent, all of its supersets will also be infrequent.

For this experiment, a minimum support of **2%** is used.

In [ ]:
MIN_SUPPORT = 0.02

frequent_itemsets = apriori(
    basket,
    min_support=MIN_SUPPORT,
    use_colnames=True
)

frequent_itemsets['itemset_size'] = frequent_itemsets['itemsets'].apply(len)

print("Number of frequent itemsets:",
      len(frequent_itemsets))

display(
    frequent_itemsets
    .sort_values('support', ascending=False)
    .head(20)
)

## 9. Frequent Itemsets by Size

Examine how many frequent itemsets were found for each itemset size.

In [ ]:
itemset_counts = (
    frequent_itemsets['itemset_size']
    .value_counts()
    .sort_index()
)

print(itemset_counts)

plt.figure(figsize=(8, 5))
itemset_counts.plot(kind='bar')
plt.title('Number of Frequent Itemsets by Size')
plt.xlabel('Itemset Size')
plt.ylabel('Number of Frequent Itemsets')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 10. Generate Association Rules

Association rules have the general form:

**A → B**

where A is the antecedent and B is the consequent.

Important measures:

### Support
How frequently A and B occur together.

### Confidence
The probability of finding B in a transaction that contains A.

**Confidence(A → B) = Support(A ∪ B) / Support(A)**

### Lift
Measures how much more often A and B occur together than would be expected if they were independent.

**Lift(A → B) = Confidence(A → B) / Support(B)**

A lift greater than 1 indicates a positive association.

In [ ]:
rules = association_rules(
    frequent_itemsets.drop(columns='itemset_size'),
    metric='confidence',
    min_threshold=0.30
)

print("Number of generated rules:", len(rules))

display(
    rules[
        ['antecedents', 'consequents',
         'support', 'confidence', 'lift']
    ]
    .sort_values('lift', ascending=False)
    .head(20)
)

## 11. Format the Association Rules

Convert the frozenset representation into readable product names.

In [ ]:
def format_itemset(itemset):
    return ', '.join(sorted(itemset))

rules_display = rules.copy()

rules_display['Antecedent'] = rules_display['antecedents'].apply(format_itemset)
rules_display['Consequent'] = rules_display['consequents'].apply(format_itemset)

rules_display = rules_display[
    ['Antecedent', 'Consequent', 'support', 'confidence', 'lift']
].sort_values(
    ['lift', 'confidence'],
    ascending=False
)

display(rules_display.head(20))

## 12. Filter Strong Association Rules

For a stronger set of rules, use:

- Confidence ≥ 50%
- Lift > 1

These conditions select rules with reasonably strong predictive association and positive lift.

In [ ]:
strong_rules = rules_display[
    (rules_display['confidence'] >= 0.50) &
    (rules_display['lift'] > 1)
].copy()

print("Number of strong rules:", len(strong_rules))

display(strong_rules.head(20))

## 13. Top Rules by Lift

Display the strongest rules according to Lift.

In [ ]:
top_rules = (
    rules_display
    .sort_values('lift', ascending=False)
    .head(15)
)

display(top_rules)

plt.figure(figsize=(10, 6))
plt.barh(
    range(len(top_rules)),
    top_rules['lift'].iloc[::-1]
)
plt.yticks(
    range(len(top_rules)),
    (
        top_rules['Antecedent'] + ' → ' +
        top_rules['Consequent']
    ).iloc[::-1]
)
plt.xlabel('Lift')
plt.title('Top Association Rules by Lift')
plt.tight_layout()
plt.show()

## 14. Top Rules by Confidence

Display rules with the highest confidence values.

In [ ]:
top_confidence_rules = (
    rules_display
    .sort_values('confidence', ascending=False)
    .head(15)
)

display(top_confidence_rules)

## 15. Example Interpretation

For a rule:

**A → B**

- **Support** tells how common A and B are together in all transactions.
- **Confidence** tells how often B occurs when A occurs.
- **Lift > 1** indicates that A and B occur together more often than expected under independence.

The exact rules and their interpretation depend on the values obtained from the dataset.

## Conclusion

The Apriori association rule mining algorithm was successfully applied to the Online Retail dataset.

The experiment included:

- Transaction preprocessing
- Removal of cancelled transactions
- Selection of frequently purchased products
- Transaction/basket creation
- One-hot encoding
- Frequent itemset generation using Apriori
- Support calculation
- Association rule generation
- Confidence calculation
- Lift calculation
- Filtering of strong rules
- Visualization and interpretation of association rules

The same Online Retail dataset has now been used across the DWM experiments for EDA, regression, classification, K-Means clustering, hierarchical clustering and association rule mining.